# Perceptron tutorial

This notebook trains **binary** `mlpackage.supervised_learning.Perceptron` (linear threshold + online updates) on sklearn **Wisconsin Breast Cancer**: malignant vs benign as labels **0** and **1**. Features are **standardized** using training statistics only.

## Prerequisites and goals

**You will**
- Use **`lr`** and **`max_iter`** for Rosenblatt-style epochs.
- Read **`training_errors`** (MSE on training data after each epoch).
- Evaluate **`score`**, **`confusion_matrix`**, and plot a **linear decision boundary** in 2D (via a small 2-feature refit).
- Compare several **`max_iter`** values on the same split.

**Dependencies:** `numpy`, `pandas`, `matplotlib`, `scikit-learn`, `mlpackage`.

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from mlpackage.supervised_learning import Perceptron

## Step 1: Load Breast Cancer data

Binary labels **0 / 1** match the perceptron’s step outputs.

In [ ]:
cancer = load_breast_cancer()
X = np.asarray(cancer.data, dtype=float)
y = np.asarray(cancer.target, dtype=int)

feature_names = list(cancer.feature_names)
target_names = list(cancer.target_names)

print("X shape:", X.shape, "  y shape:", y.shape)
print("Classes:", {i: target_names[i] for i in range(2)})
print("Label counts:", dict(zip(*np.unique(y, return_counts=True))))

## Step 2: Stratified train/test split

In [ ]:
RANDOM_STATE = 42
TEST_FRACTION = 0.30

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_FRACTION,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Train:", X_train.shape[0], "  Test:", X_test.shape[0])

## Step 3: Standardize (train-only fit)

In [ ]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

## Step 4: Fit `Perceptron`

Online updates use **`lr`**; **`max_iter`** is the number of **epochs** (full passes). Start modestly—non-separable data can make errors oscillate.

In [ ]:
LR = 0.01
MAX_ITER = 100

clf = Perceptron(lr=LR, max_iter=MAX_ITER)
clf.fit(X_train_s, y_train)

print("Epochs recorded:", len(clf.training_errors))
print("Final training MSE:", round(clf.training_errors[-1], 6))

## Step 5: Predictions and accuracy

Outputs are hard **0/1** labels (no probabilities).

In [ ]:
y_pred_test = clf.predict(X_test_s)
y_pred_train = clf.predict(X_train_s)

print(f"Train accuracy: {clf.score(X_train_s, y_train):.4f}")
print(f"Test accuracy : {clf.score(X_test_s, y_test):.4f}")

n_show = 12
prev = pd.DataFrame(
    {
        "y_true": [target_names[t] for t in y_test[:n_show]],
        "y_pred": [target_names[t] for t in y_pred_test[:n_show]],
    }
)
display(prev)

## Step 6: Confusion matrix, per-class counts, and training MSE curve

After each epoch, **`training_errors[k]`** stores training-set MSE. Use it to see whether the online procedure is still changing predictions.

In [ ]:
display(clf.confusion_matrix(X_test_s, y_test))

for c in (0, 1):
    mask = y_test == c
    hits = int(np.sum((y_pred_test == y_test) & mask))
    tot = int(np.sum(mask))
    print(f"Test true {target_names[c]}: {hits}/{tot} correct")

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(clf.training_errors, linewidth=1.2)
plt.xlabel("Epoch (after each full pass over training rows)")
plt.ylabel("Training MSE")
plt.title("Perceptron — training error vs epoch")
plt.grid(True, alpha=0.3)
plt.tight_layout()

_nb = Path("perceptron_tutorial.ipynb")
_p = (
    _nb.with_name("breast_cancer_perceptron_training_mse.png")
    if _nb.is_file()
    else Path(
        "examples/supervised_learning/perceptron/breast_cancer_perceptron_training_mse.png"
    )
)
_p.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(_p, dpi=150, bbox_inches="tight")
print(f"Saved figure to: {_p.resolve()}")
plt.show()

## Step 7: Linear decision boundary (2-feature model, test set)

The fitted classifier in Step 4 uses **30** features. We **refit** a `Perceptron` on **mean radius** and **mean texture** only to draw the line \( \mathbf{w}^\top \mathbf{x} + b = 0 \) and shade the two half-planes.

In [ ]:
i, j = 0, 1
X_train_2d = X_train_s[:, [i, j]]
X_test_2d = X_test_s[:, [i, j]]

viz = Perceptron(lr=LR, max_iter=MAX_ITER)
viz.fit(X_train_2d, y_train)

pad = 0.6
h = 0.02
x_min = float(min(X_train_2d[:, 0].min(), X_test_2d[:, 0].min()) - pad)
x_max = float(max(X_train_2d[:, 0].max(), X_test_2d[:, 0].max()) + pad)
y_min = float(min(X_train_2d[:, 1].min(), X_test_2d[:, 1].min()) - pad)
y_max = float(max(X_train_2d[:, 1].max(), X_test_2d[:, 1].max()) + pad)

xx, yy = np.meshgrid(
    np.arange(x_min, x_max, h),
    np.arange(y_min, y_max, h),
)
grid = np.c_[xx.ravel(), yy.ravel()]
Z = viz.predict(grid).reshape(xx.shape)

plt.figure(figsize=(8, 6))
plt.contourf(
    xx,
    yy,
    Z,
    levels=[-0.5, 0.5, 1.5],
    colors=["#ffcccc", "#cce5ff"],
    alpha=0.95,
)

w2, b2 = viz.coef_, viz.intercept_
xs_line = np.linspace(x_min, x_max, 200)
if abs(w2[1]) > 1e-12:
    ys_line = -(w2[0] * xs_line + b2) / w2[1]
    plt.plot(xs_line, ys_line, "k-", linewidth=2.0, label="Decision boundary")
else:
    plt.axvline(-b2 / w2[0], color="k", linewidth=2.0, label="Decision boundary")

m0 = y_test == 0
m1 = y_test == 1
plt.scatter(
    X_test_2d[m0, 0],
    X_test_2d[m0, 1],
    c="red",
    marker="s",
    s=55,
    edgecolors="black",
    linewidths=0.6,
    label=f"{target_names[0]} (test)",
    zorder=5,
)
plt.scatter(
    X_test_2d[m1, 0],
    X_test_2d[m1, 1],
    c="blue",
    marker="x",
    s=85,
    linewidths=1.6,
    label=f"{target_names[1]} (test)",
    zorder=5,
)

plt.xlabel(f"Feature 1 — {feature_names[i]} [standardized]")
plt.ylabel(f"Feature 2 — {feature_names[j]} [standardized]")
plt.title("Perceptron decision boundary (2-feature model, test set)")
plt.legend(loc="upper right", fontsize=9)
plt.xlim(x_min, x_max)
plt.ylim(y_min, y_max)
plt.tight_layout()

_nb_here = Path("perceptron_tutorial.ipynb")
_plot_path = (
    _nb_here.with_name("breast_cancer_perceptron_decision_boundary.png")
    if _nb_here.is_file()
    else Path(
        "examples/supervised_learning/perceptron/breast_cancer_perceptron_decision_boundary.png"
    )
)
_plot_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(_plot_path, dpi=150, bbox_inches="tight")
print(f"Saved figure to: {_plot_path.resolve()}")
print("2-feature test accuracy:", round(viz.score(X_test_2d, y_test), 4))
plt.show()

## Step 8: Compare `max_iter` on the same split

More epochs do not always improve **test** accuracy—online updates can oscillate when classes overlap.

In [ ]:
candidates = [25, 50, 100, 200]
rows = []

for mi in candidates:
    m = Perceptron(lr=LR, max_iter=mi)
    m.fit(X_train_s, y_train)
    rows.append(
        {
            "max_iter": mi,
            "train_acc": m.score(X_train_s, y_train),
            "test_acc": m.score(X_test_s, y_test),
            "final_train_mse": m.training_errors[-1],
        }
    )

display(pd.DataFrame(rows).round(6))